In [ ]:
import mysql.connector as mc 
from mysql.connector import Error
import dotenv
import os
from datetime import datetime, timedelta

dotenv.load_dotenv(override=True)

DB_CONFIG = {
        'host': os.getenv('DB_HOST'),
        'port': int(os.getenv('DB_PORT', 3306)),
        'user': os.getenv('DB_USER'),
        'password': os.getenv('DB_PASSWORD'),
        'database': os.getenv('DB_DATABASE'),
        'autocommit': False,
}
print_config = {key: DB_CONFIG[key] if key != 'password' else '****' for key in DB_CONFIG}
print(f"[INFO] DB_CONFIG={print_config}")

paths = [
    './노드정보.csv',
    './도로정보.csv',
    './돌발상황.csv',
]


In [ ]:
import csv

f = open(paths[0], "r", encoding='cp949')
reader = csv.reader(f)

for row in reader:
    print(row)


In [ ]:
create_query="""
CREATE TABLE traffic_node (
    node_id        BIGINT PRIMARY KEY,
    node_type      VARCHAR(50),
    node_name      VARCHAR(100),
    turn_restrict  CHAR(1),
    node_name_en   VARCHAR(100),
    x_coord        DECIMAL(10,6),
    y_coord        DECIMAL(10,6),
    prev_node_id   BIGINT,
    updated_date   DATE,
    remark         VARCHAR(255)
);
"""
conn = mc.connect(**DB_CONFIG)
cursor = conn.cursor()
cursor.execute(create_query)

In [ ]:
BATCH_SIZE = 500

def to_int(v):
    return int(v) if v and v.strip() else None

def to_float(v):
    return float(v) if v and v.strip() else None

def to_str(v):
    return v.strip() if v and v.strip() else None


def process_csv_and_insert(cursor, conn, file_path):
    batch = []
    total = 0
    success = 0
    fail = 0

    with open(file_path, "r", encoding="cp949") as f:
        reader = csv.reader(f)
        next(reader)

        for i, row in enumerate(reader, start=1):
            total += 1
            try:
                # 필수값 검증 (필요시)
                if not row[0] or not row[0].strip():
                    raise ValueError("node_id is required")

                data = (
                    to_int(row[0]),
                    to_str(row[1]),
                    to_str(row[2]),
                    to_str(row[3]),
                    to_str(row[4]),
                    to_float(row[5]),
                    to_float(row[6]),
                    to_int(row[7]),
                    to_str(row[8]),
                    to_str(row[9]),
                )

                batch.append(data)

            except Exception as e:
                fail += 1
                print(f"[SKIP] row {i}: {e} | raw={row}")
                continue

            # 배치 꽉 차면 insert
            if len(batch) >= BATCH_SIZE:
                insert_batch(cursor, conn, batch)
                success += len(batch)
                batch.clear()

        # 마지막 남은 배치 처리
        if batch:
            insert_batch(cursor, conn, batch)
            success += len(batch)

    print(f"""
    ===== RESULT =====
    total   : {total}
    success : {success}
    fail    : {fail}
    ==================
    """)


def insert_batch(cursor, conn, batch):
    query = """
        INSERT INTO traffic_node (
            node_id, node_type, node_name, turn_restrict,
            node_name_en, x_coord, y_coord,
            prev_node_id, updated_date, remark
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    try:
        cursor.executemany(query, batch)
        conn.commit()
    except Exception as e:
        print(f"[BATCH ERROR] {e}")
        conn.rollback()

conn = mc.connect(**DB_CONFIG)
cursor = conn.cursor()
process_csv_and_insert(cursor, conn, paths[0])
